# `sql` — Reference

Run DuckDB SQL directly over a DataFrame. The current frame is registered as table **`data`**.
Optional extra: `pip install 'pytae[sql]'` (pulls in `duckdb`).

| Call style | Meaning |
|---|---|
| `pt.sql(df, query)` | Function form; the passed frame is table `data` |
| `df.pt.sql(query)` | Accessor form; chains with pandas / other `pt` methods |
| `pt.sql(df, query, extra=other)` | Register extra frames as additional tables (like CLI `-file` aliases) |

The name `data` is reserved for the current frame — an extra frame cannot be named `data`.
Column names with spaces need double quotes inside the query: `"bill length mm"`.

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pandas as pd
import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']
titanic = pt.sample_data['titanic']

## Function form — current frame is table `data`

In [2]:
# The passed DataFrame is registered as the table `data`
pt.sql(
    penguins,
    'select species, avg(body_mass_g) as avg_mass, count(*) as n '
    'from data group by species order by species'
)

,species,avg_mass,n
0,Adelie,3700.662252,152
1,Chinstrap,3733.088235,68
2,Gentoo,5076.016260,124


## Accessor form — `df.pt.sql(...)`

Same query engine, but chains naturally off any DataFrame.

In [3]:
# `data` inside the query refers to the frame the accessor is called on
(
    penguins
    .pt.sql("select * from data where species = 'Adelie'")
    .head()
)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


## Spaced column names — double quotes, brackets `[col]`, or backticks

SQL standard uses double quotes for identifiers with spaces. `pytae` also accepts SQL-style square brackets `[col a]` and backticks (both converted to standard SQL double quotes before querying DuckDB):

In [4]:
spaced = pd.DataFrame({'col a': [1, 20, 3], 'col b': ['x', 'y', 'z']})

# Double quotes, brackets, and backticks all work
pt.sql(spaced, 'select [col a], `col b` from data where [col a] > 2')

,col a,col b
0,20,y
1,3,z


## Extra frames — joins across DataFrames

Pass additional DataFrames as keyword arguments; each becomes a table of that name.

In [5]:
left = pd.DataFrame({'id': [1, 2, 3], 'x': ['a', 'b', 'c']})
right = pd.DataFrame({'id': [1, 3], 'y': ['p', 'q']})

# `left` is `data`; `right` is registered under the keyword name `extra`
pt.sql(left, 'select data.id, x, y from data inner join extra using (id)', extra=right)

,id,x,y
0,1,a,p
1,3,c,q


## Chaining — mix SQL with pandas and other `pt` methods

In [6]:
# SQL to aggregate, then a plain pandas method, then another pt accessor call
(
    tips
    .pt.sql('select day, time, sum(total_bill) as bill, sum(tip) as tip from data group by day, time')
    .sort_values('bill', ascending=False)
    .pt.mutate(tip_pct='tip / bill * 100')
)


,day,time,bill,tip,tip_pct
4,Sat,Dinner,1778.40,260.40,14.642375
5,Sun,Dinner,1627.16,247.39,15.203791
0,Thur,Lunch,1077.55,168.83,15.667950
3,Fri,Dinner,235.96,35.28,14.951687
2,Fri,Lunch,89.92,16.68,18.549822
1,Thur,Dinner,18.78,3.00,15.974441


## Loading queries from an external file — `@query.txt`

Just like the CLI (`pytae data.parquet -sql @query.txt`), you can pass `@path.txt` to execute SQL queries directly from a file:

In [7]:
with open('query_demo.txt', 'w') as f:
    f.write('select species, avg(body_mass_g) as avg_mass from data group by species order by avg_mass desc')

out = penguins.pt.sql('@query_demo.txt')
if os.path.exists('query_demo.txt'):
    os.remove('query_demo.txt')
out

,species,avg_mass
0,Gentoo,5076.016260
1,Chinstrap,3733.088235
2,Adelie,3700.662252
